# Recertification KYC — Extraction automatique CIN / Permis / Passeport / Justificatif de domicile

## Objectif

Pipeline de **recertification KYC** à partir d'un zip contenant, pour chaque client (dossier nommé par son `id`), des PDFs scannés (souvent de mauvaise qualité, parfois issus de photocopies, avec rotation variable).
On extrait uniquement deux documents par dossier client :

- `JUSTIFICATIF IDENTITE.PDF` (CIN, permis de conduire biométrique, ou passeport)
- `JUSTIFICATIF DOMICILE.PDF` (justificatif de domicile)

## Logique générale

1. Dézipper et parcourir les dossiers clients.
2. Convertir les PDFs en images, corriger la rotation, nettoyer l'image (débruitage / contraste) pour compenser la mauvaise qualité des scans/photocopies.
3. **Détecter la présence d'une zone MRZ (Machine Readable Zone)** :
   - Si MRZ présente → extraction via **PassportEye** (librairie de lecture MRZ) + parsing manuel selon les règles ICAO données par l'utilisateur (TD1 3 lignes pour CIN/permis, TD3 2 lignes pour passeport).
   - Si pas de MRZ → OCR via le modèle **PaddleOCR-VL** chargé localement depuis le ModelHub Domino (`/domino/edv/modelhub/ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL/main`).
4. Détecter le pays d'émission du document.
   - Si document algérien → utiliser un **schéma d'extraction dédié par type de document** (CIN, permis, justificatif domicile), bilingue arabe/français.
   - Sinon → extraction générique sans schéma (best-effort, basée sur des heuristiques de dates/regex).
5. Construire un tableau consolidé des informations extraites par client.
6. Charger `tiers.csv`, normaliser les noms/prénoms (ordre nom-prénom ou prénom-nom inconnu) et les dates (`YYYY-MM-DD`).
7. Faire correspondre (`join`) les deux jeux de données par `id` et calculer les **incohérences** (nom/prénom, date de naissance, date d'expiration).
8. Exporter un rapport d'incohérences.

> ⚠️ **Notes importantes avant exécution**
> - Ce notebook suppose que Tesseract OCR et Poppler (pour `pdf2image`) sont installés au niveau système (voir cellule d'installation ci-dessous).
> - Les schémas algériens (regex FR/AR) sont fournis à titre de base de départ : à ajuster avec de vrais échantillons (mise en page recto/verso CIN, permis, factures Sonelgaz/Algérie Télécom pour justificatif de domicile, etc.).
> - Le chargement du modèle PaddleOCR-VL suppose que le chemin ModelHub contient un snapshot HuggingFace complet (config, poids, tokenizer/processor) — comme c'est le cas pour un chargement `AutoModel.from_pretrained(local_path, trust_remote_code=True)`.
> - Tout le pipeline est écrit de façon défensive (try/except + logs) car la qualité des scans est hétérogène : mieux vaut una ligne "à vérifier manuellement" qu'un plantage du batch.


## 1. Installation des dépendances

### Dépendances système (à exécuter une fois, nécessite les droits appropriés sur l'environnement Domino)

```bash
sudo apt-get update
sudo apt-get install -y tesseract-ocr poppler-utils libgl1
```

- `tesseract-ocr` : requis par `pytesseract` et par `passporteye` (qui l'utilise en interne pour OCRiser la zone MRZ détectée).
- `poppler-utils` : requis par `pdf2image` pour convertir les PDF en images.
- `libgl1` : requis par `opencv-python` sur certains environnements headless.

### Dépendances Python (versions figées)


In [ ]:
# Décommenter si l'environnement Domino ne les a pas déjà (recommandé de figer les versions)
# %pip install --quiet \
#     PyMuPDF==1.24.11 \
#     pdf2image==1.17.0 \
#     opencv-python-headless==4.10.0.84 \
#     numpy==1.26.4 \
#     pandas==2.2.3 \
#     Pillow==10.4.0 \
#     pytesseract==0.3.13 \
#     passporteye==2.2.2 \
#     python-mrz==0.6.2 \
#     rapidfuzz==3.10.0 \
#     Unidecode==1.3.8 \
#     python-dateutil==2.9.0.post0 \
#     tqdm==4.66.5 \
#     openpyxl==3.1.5 \
#     torch==2.4.1 \
#     transformers==4.46.2 \
#     accelerate==0.34.2 \
#     safetensors==0.4.5 \
#     einops==0.8.0 \
#     sentencepiece==0.2.0 \
#     tiktoken==0.8.0

print("Dépendances : décommenter la cellule ci-dessus lors de la première exécution.")


## 2. Imports et configuration globale

In [ ]:
import os
import re
import io
import json
import shutil
import logging
import zipfile
import unicodedata
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import fitz  # PyMuPDF
from pdf2image import convert_from_path
import pytesseract

from rapidfuzz import fuzz
from unidecode import unidecode
from dateutil import parser as dateparser
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("kyc_recert")


In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION — à adapter à l'environnement Domino
# ---------------------------------------------------------------------------

# Zip contenant les dossiers clients (un sous-dossier par client, nommé par son id)
ZIP_PATH = "/mnt/data/dossiers_clients.zip"

# Répertoire de travail où le zip est extrait
WORKDIR = Path("/home/claude/kyc_work")
EXTRACT_DIR = WORKDIR / "extracted"
DEBUG_DIR = WORKDIR / "debug_images"          # images intermédiaires (rotation corrigée, MRZ crop, etc.)
OUTPUT_DIR = WORKDIR / "outputs"

for d in (WORKDIR, EXTRACT_DIR, DEBUG_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Fichier référentiel tiers
TIERS_CSV_PATH = "/mnt/data/tiers.csv"

# Noms exacts (insensibles à la casse/accents) des PDFs à traiter dans chaque dossier client
TARGET_DOC_IDENTITE = "JUSTIFICATIF IDENTITE.PDF"
TARGET_DOC_DOMICILE = "JUSTIFICATIF DOMICILE.PDF"

# Chemin local du modèle PaddleOCR-VL sur le ModelHub Domino (external data volume)
PADDLEOCR_VL_LOCAL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL/main"

# Device pour le modèle VLM de fallback (mettre "cuda" si un GPU est disponible dans le compute Domino)
MODEL_DEVICE = "cuda" if os.environ.get("USE_GPU", "0") == "1" else "cpu"

# DPI de rendu des PDF -> images (plus haut = plus net, mais plus lent ; 300 est un bon compromis pour la MRZ)
PDF_RENDER_DPI = 300

print("Configuration chargée.")


## 3. Extraction du zip et listing des dossiers clients

Chaque dossier client est nommé par son `id` (celui qui servira de clé de jointure avec `tiers.csv`).

In [ ]:
def extract_zip(zip_path: str, extract_dir: Path) -> Path:
    '''Dézippe le fichier client vers extract_dir (idempotent).'''
    zip_path = Path(zip_path)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip introuvable : {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    logger.info(f"Zip extrait dans {extract_dir}")
    return extract_dir


def list_client_folders(extract_dir: Path) -> List[Path]:
    '''Liste les dossiers clients (un niveau, ou détecte automatiquement le niveau
    si le zip contient un dossier racine unique enveloppant tous les clients).'''
    entries = [p for p in extract_dir.iterdir() if p.is_dir() and not p.name.startswith("__MACOSX")]

    # Cas où le zip contient un seul dossier racine qui contient lui-même les dossiers clients
    if len(entries) == 1 and any(c.is_dir() for c in entries[0].iterdir()):
        sub_entries = [p for p in entries[0].iterdir() if p.is_dir()]
        if len(sub_entries) > 1:
            entries = sub_entries

    logger.info(f"{len(entries)} dossiers clients détectés.")
    return entries


# Décommenter pour exécuter réellement :
# extract_zip(ZIP_PATH, EXTRACT_DIR)
# client_folders = list_client_folders(EXTRACT_DIR)
# client_folders[:5]


## 4. Repérage des PDFs cibles (`JUSTIFICATIF IDENTITE.PDF` / `JUSTIFICATIF DOMICILE.PDF`)

Les noms de fichiers peuvent varier légèrement en casse, accents ou espaces selon les dossiers ; la recherche est donc normalisée.

In [ ]:
def _normalize_filename(name: str) -> str:
    name = unidecode(name).upper()
    name = re.sub(r"\s+", " ", name).strip()
    return name


def find_target_pdfs(client_folder: Path) -> Dict[str, Optional[Path]]:
    '''Retrouve, dans un dossier client, le PDF d'identité et le PDF de domicile,
    en tolérant variations de casse/accents/espaces dans le nom de fichier.'''
    result = {"identite": None, "domicile": None}

    target_identite_norm = _normalize_filename(TARGET_DOC_IDENTITE)
    target_domicile_norm = _normalize_filename(TARGET_DOC_DOMICILE)

    for pdf_path in client_folder.rglob("*.pdf"):
        norm = _normalize_filename(pdf_path.name)
        if norm == target_identite_norm:
            result["identite"] = pdf_path
        elif norm == target_domicile_norm:
            result["domicile"] = pdf_path

    if result["identite"] is None:
        logger.warning(f"[{client_folder.name}] JUSTIFICATIF IDENTITE.PDF introuvable.")
    if result["domicile"] is None:
        logger.warning(f"[{client_folder.name}] JUSTIFICATIF DOMICILE.PDF introuvable.")

    return result


## 5. Conversion PDF → images et prétraitement

Les scans étant souvent des **photocopies de mauvaise qualité** avec une **rotation variable**, on applique :

1. Rendu du PDF en image haute résolution (`PDF_RENDER_DPI`).
2. **Correction de l'orientation** de la page (0°/90°/180°/270°) via l'OSD (*Orientation and Script Detection*) de Tesseract — plus robuste que le deskew seul quand la page est tournée à 90°/180°/270°.
3. **Deskew fin** (petite inclinaison résiduelle) via détection de contours/`minAreaRect`.
4. **Nettoyage** : passage en niveaux de gris, débruitage (`fastNlMeansDenoising`), amélioration du contraste (CLAHE), et léger *sharpening* — utile pour compenser l'effet photocopie (contraste faible, bruit poivre-et-sel).

In [ ]:
def pdf_to_images(pdf_path: Path, dpi: int = PDF_RENDER_DPI) -> List[np.ndarray]:
    '''Convertit toutes les pages d'un PDF en images OpenCV (BGR).'''
    pages = convert_from_path(str(pdf_path), dpi=dpi)
    images = [cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR) for p in pages]
    return images


def correct_page_orientation(image: np.ndarray) -> np.ndarray:
    '''Corrige une rotation grossière de 90/180/270° via l'OSD Tesseract.
    Retombe sur l'image d'origine si l'OSD échoue (image trop bruitée / trop peu de texte).'''
    try:
        osd = pytesseract.image_to_osd(image, output_type=pytesseract.Output.DICT)
        rotate_angle = int(osd.get("rotate", 0))
        if rotate_angle != 0:
            # cv2.rotate ne gère que des multiples de 90° -> on utilise warpAffine générique
            (h, w) = image.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, -rotate_angle, 1.0)
            # Recalcule la taille du canevas pour ne pas rogner l'image après rotation
            cos, sin = abs(M[0, 0]), abs(M[0, 1])
            new_w = int((h * sin) + (w * cos))
            new_h = int((h * cos) + (w * sin))
            M[0, 2] += (new_w / 2) - center[0]
            M[1, 2] += (new_h / 2) - center[1]
            image = cv2.warpAffine(image, M, (new_w, new_h), flags=cv2.INTER_CUBIC,
                                    borderMode=cv2.BORDER_REPLICATE)
            logger.info(f"Rotation grossière corrigée : {rotate_angle}°")
    except pytesseract.TesseractError as e:
        logger.warning(f"OSD Tesseract impossible (image probablement trop bruitée) : {e}")
    return image


def deskew_fine(image: np.ndarray) -> np.ndarray:
    '''Correction fine d'une petite inclinaison résiduelle (quelques degrés).'''
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.bitwise_not(gray)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if coords.shape[0] < 50:
        return image  # pas assez de pixels de texte pour une estimation fiable
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 0.3 or abs(angle) > 15:
        return image  # ignore si négligeable ou aberrant (probable mauvaise estimation)
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def clean_scan(image: np.ndarray) -> np.ndarray:
    '''Nettoyage pour compenser un scan/photocopie de mauvaise qualité.
    Retourne une image BGR nettoyée (toujours 3 canaux pour rester compatible
    avec les moteurs OCR/VLM qui attendent du RGB/BGR).'''
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10, templateWindowSize=7, searchWindowSize=21)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrasted = clahe.apply(denoised)
    sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    sharpened = cv2.filter2D(contrasted, -1, sharpen_kernel)
    return cv2.cvtColor(sharpened, cv2.COLOR_GRAY2BGR)


def preprocess_page(image: np.ndarray) -> np.ndarray:
    '''Pipeline complet de prétraitement d'une page scannée.'''
    image = correct_page_orientation(image)
    image = deskew_fine(image)
    image = clean_scan(image)
    return image


def load_and_preprocess_pdf(pdf_path: Path) -> List[np.ndarray]:
    '''Convertit + prétraite toutes les pages d'un PDF. Un document d'identité recto/verso
    ou multi-pages produit plusieurs images ; on les traite toutes puis on fusionne
    les informations trouvées.'''
    pages = pdf_to_images(pdf_path)
    return [preprocess_page(p) for p in pages]


## 6. Détection de la présence d'une MRZ et lecture via PassportEye

`PassportEye` localise automatiquement la bande MRZ (zone jaune sur l'exemple fourni) sur l'image et l'OCRise
avec un moteur Tesseract dédié aux caractères MRZ (`OCRB`). On l'utilise ici en **deux temps** :

1. **Détection** : `read_mrz` renvoie `None` si aucune zone MRZ plausible n'est trouvée → le document est traité
   comme "sans MRZ" (fallback PaddleOCR-VL, cf. section 8).
2. **Lecture brute des lignes MRZ** (`mrz_lines`) que l'on reparse **nous-mêmes** avec les règles exactes données
   (positions de caractères), car les règles ICAO standard de `PassportEye`/`mrz` sont parfois trop strictes
   (chiffres de contrôle) face à des OCR de mauvaise qualité sur photocopies.

### Rappel des règles fournies

**CIN / permis biométrique algérien (MRZ sur 3 lignes, format TD1)** :
- Ligne 2 : caractères 1-6 = date de naissance (`YYMMDD`) ; caractères juste après le sexe (`M`/`F`) sur 6
  caractères = date d'expiration (`YYMMDD`).
- Ligne 3 : nom = chaîne avant le premier `<` ; prénom = chaîne entre `<<` et la série de `<<<<...` finale.

**Passeport biométrique algérien (MRZ sur 2 lignes, format TD3)** :
- Ligne 1 : nom = chaîne après `DZA` et avant `<` ; prénom = chaîne entre `<<` et `<<<<...`.
- Ligne 2 : 6 caractères après `DZA` = date de naissance (`YYMMDD`) ; 6 caractères après le sexe (`M`/`F`) =
  date d'expiration (`YYMMDD`).

Les deux formats sont différenciés par le **nombre de lignes MRZ détectées** (3 → TD1 CIN/permis, 2 → TD3 passeport).

In [ ]:
from passporteye import read_mrz


def detect_and_read_mrz(image: np.ndarray, debug_name: str = "doc") -> Optional[Dict]:
    '''Tente de détecter + OCRiser la zone MRZ sur une image de page.
    Retourne None si aucune MRZ n'est détectée (document 'sans MRZ').'''
    try:
        # PassportEye accepte un chemin de fichier ou un tableau numpy (via un buffer temporaire)
        tmp_path = DEBUG_DIR / f"_mrz_input_{debug_name}.png"
        cv2.imwrite(str(tmp_path), image)
        mrz = read_mrz(str(tmp_path), save_roi=True)
        if mrz is None:
            return None

        raw_lines = [l for l in mrz.aux.get("text", "").split("\n") if l.strip()] if hasattr(mrz, "aux") else []
        if not raw_lines:
            # fallback : reconstruit depuis les champs structurés si le texte brut n'est pas exposé
            raw_lines = getattr(mrz, "mrz_lines", []) or []

        if not raw_lines:
            return None

        return {"lines": raw_lines, "raw_object": mrz}
    except Exception as e:
        logger.warning(f"[{debug_name}] Détection MRZ échouée : {e}")
        return None


def has_machine_readable_zone(pages: List[np.ndarray], debug_name: str = "doc") -> Tuple[bool, Optional[Dict]]:
    '''Parcourt toutes les pages du document (recto/verso) et retourne (True, mrz_result)
    dès qu'une MRZ plausible est trouvée sur l'une des pages.'''
    for i, page in enumerate(pages):
        result = detect_and_read_mrz(page, debug_name=f"{debug_name}_p{i}")
        if result is not None and len(result["lines"]) >= 2:
            return True, result
    return False, None


## 7. Parsing manuel des lignes MRZ (règles exactes CIN/permis TD1 et passeport TD3)

In [ ]:
def _mrz_yy_mm_dd_to_iso(yy_mm_dd: str, is_expiry: bool = False) -> Optional[str]:
    '''Convertit une date MRZ YYMMDD -> YYYY-MM-DD.
    Règle de siècle : pour une date de naissance, on suppose 19xx si YY > (année courante - 2000) + marge,
    sinon 20xx ; pour une date d'expiration, on suppose toujours 20xx (documents en cours de validité).'''
    if not yy_mm_dd or len(yy_mm_dd) != 6 or not yy_mm_dd.isdigit():
        return None
    yy, mm, dd = yy_mm_dd[0:2], yy_mm_dd[2:4], yy_mm_dd[4:6]
    if is_expiry:
        century = "20"
    else:
        current_yy = datetime.now().year % 100
        century = "20" if int(yy) <= current_yy + 1 else "19"
    try:
        full_year = int(century + yy)
        date_obj = datetime(full_year, int(mm), int(dd))
        return date_obj.strftime("%Y-%m-%d")
    except ValueError:
        return None


def _clean_mrz_names(name_field: str) -> Tuple[Optional[str], Optional[str]]:
    '''Sépare nom / prénom d'un champ nom MRZ du type 'EL<HACHIMI<<SAID<<<<<<<<<<<<<'
    -> nom = chaîne avant le premier '<' isolé (avant '<<')
    -> prénom = chaîne entre '<<' et la série finale de '<'.'''
    name_field = name_field.strip()
    if "<<" not in name_field:
        return None, None
    nom_part, remainder = name_field.split("<<", 1)
    prenom_part = remainder.split("<<<")[0].split("<<")[0]  # on coupe avant tout retour à '<<<...'
    prenom_part = re.sub(r"<+$", "", prenom_part)
    nom = nom_part.replace("<", " ").strip()
    prenom = prenom_part.replace("<", " ").strip()
    return (nom or None), (prenom or None)


def parse_mrz_td1(lines: List[str]) -> Dict:
    '''CIN / permis biométrique algérien — MRZ sur 3 lignes (format TD1).
    Ligne 2 (index 1) : 6 premiers caractères = date naissance ; 6 caractères après le M/F = date expiration.
    Ligne 3 (index 2) : NOM<<PRENOM<<<<<...'''
    result = {"nom": None, "prenom": None, "date_naissance": None, "date_expiration": None,
              "doc_format": "TD1", "pays_mrz": None}
    if len(lines) < 3:
        return result

    l1, l2, l3 = lines[0], lines[1], lines[2]

    # Pays d'émission : caractères 3-5 de la ligne 1 (ex : "IDDZA..." -> DZA)
    m_country = re.match(r"^[A-Z0-9<]{2}([A-Z]{3})", l1)
    if m_country:
        result["pays_mrz"] = m_country.group(1)

    l2_clean = l2.replace(" ", "")
    dob_raw = l2_clean[0:6]
    result["date_naissance"] = _mrz_yy_mm_dd_to_iso(dob_raw, is_expiry=False)

    m_sex = re.search(r"[MF<]", l2_clean[6:8])
    if m_sex:
        sex_pos = 6 + m_sex.start()
        expiry_raw = l2_clean[sex_pos + 1: sex_pos + 7]
        result["date_expiration"] = _mrz_yy_mm_dd_to_iso(expiry_raw, is_expiry=True)

    nom, prenom = _clean_mrz_names(l3)
    result["nom"], result["prenom"] = nom, prenom
    return result


def parse_mrz_td3(lines: List[str]) -> Dict:
    '''Passeport biométrique algérien — MRZ sur 2 lignes (format TD3).
    Ligne 1 : P<DZA + NOM<<PRENOM<<<<<...  (nom après le code pays DZA, avant '<')
    Ligne 2 : n° passeport + code pays DZA + date naissance (6c) + sexe + date expiration (6c).'''
    result = {"nom": None, "prenom": None, "date_naissance": None, "date_expiration": None,
              "doc_format": "TD3", "pays_mrz": None}
    if len(lines) < 2:
        return result

    l1, l2 = lines[0], lines[1]

    m_country = re.search(r"DZA", l1)
    if m_country:
        result["pays_mrz"] = "DZA"
        name_zone = l1[m_country.end():]
        nom, prenom = _clean_mrz_names(name_zone)
        result["nom"], result["prenom"] = nom, prenom

    l2_clean = l2.replace(" ", "")
    m_country2 = re.search(r"DZA", l2_clean)
    if m_country2:
        dob_raw = l2_clean[m_country2.end(): m_country2.end() + 6]
        result["date_naissance"] = _mrz_yy_mm_dd_to_iso(dob_raw, is_expiry=False)

        m_sex = re.search(r"[MF<]", l2_clean[m_country2.end() + 6: m_country2.end() + 8])
        if m_sex:
            sex_pos = m_country2.end() + 6 + m_sex.start()
            expiry_raw = l2_clean[sex_pos + 1: sex_pos + 7]
            result["date_expiration"] = _mrz_yy_mm_dd_to_iso(expiry_raw, is_expiry=True)

    return result


def parse_mrz(mrz_result: Dict) -> Dict:
    '''Choisit le bon parseur selon le nombre de lignes MRZ détectées
    (3 lignes -> CIN/permis TD1 ; 2 lignes -> passeport TD3).'''
    lines = [l.strip() for l in mrz_result["lines"] if l.strip()]
    if len(lines) >= 3:
        parsed = parse_mrz_td1(lines[-3:])   # les 3 dernières lignes utiles si du bruit précède
        parsed["type_document_probable"] = "CIN_ou_PERMIS"
    else:
        parsed = parse_mrz_td3(lines[-2:])
        parsed["type_document_probable"] = "PASSEPORT"
    parsed["mrz_lines_brutes"] = lines
    return parsed


## 8. Documents sans MRZ : OCR via PaddleOCR-VL (modèle local ModelHub Domino)

Pour les documents **sans** zone MRZ (beaucoup de cartes non biométriques, permis anciens, justificatifs de
domicile, cartes/passeports étrangers non biométriques, etc.), on utilise le modèle **PaddleOCR-VL** chargé
**directement depuis le chemin local du ModelHub Domino** (external data volume), sans appel réseau :

```
/domino/edv/modelhub/ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL/main
```

Le dossier étant un snapshot HuggingFace complet (config + poids + processor), on le charge avec
`AutoModelForCausalLM` / `AutoProcessor` (`trust_remote_code=True`, car PaddleOCR-VL embarque du code
personnalisé). Le modèle est chargé **une seule fois** (singleton), puis réutilisé pour tous les documents.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

_paddleocr_vl_model = None
_paddleocr_vl_processor = None


def load_paddleocr_vl(local_path: str = PADDLEOCR_VL_LOCAL_PATH, device: str = MODEL_DEVICE):
    '''Charge (une seule fois) le modèle PaddleOCR-VL depuis le chemin local du ModelHub.
    Ne fait AUCUN appel réseau : local_files_only=True force l'utilisation exclusive du snapshot local.'''
    global _paddleocr_vl_model, _paddleocr_vl_processor

    if _paddleocr_vl_model is not None:
        return _paddleocr_vl_model, _paddleocr_vl_processor

    local_path = Path(local_path)
    if not local_path.exists():
        raise FileNotFoundError(
            f"Chemin ModelHub introuvable : {local_path}. "
            f"Vérifier que le external data volume Domino est bien monté."
        )

    logger.info(f"Chargement de PaddleOCR-VL depuis {local_path} (device={device})...")

    _paddleocr_vl_processor = AutoProcessor.from_pretrained(
        str(local_path), trust_remote_code=True, local_files_only=True
    )
    _paddleocr_vl_model = AutoModelForCausalLM.from_pretrained(
        str(local_path),
        trust_remote_code=True,
        local_files_only=True,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device)
    _paddleocr_vl_model.eval()

    logger.info("PaddleOCR-VL chargé.")
    return _paddleocr_vl_model, _paddleocr_vl_processor


# Prompt générique demandant une transcription brute et fidèle (pas de correction/interprétation),
# pour préserver le mélange arabe/français des documents algériens.
_OCR_PROMPT = (
    "Transcris fidèlement tout le texte visible sur ce document, ligne par ligne, "
    "sans traduire ni corriger, en conservant le mélange de langues (arabe/français) tel quel."
)


def ocr_page_with_paddleocr_vl(image: np.ndarray, device: str = MODEL_DEVICE, prompt: str = _OCR_PROMPT) -> str:
    '''OCRise une page (image OpenCV BGR) avec PaddleOCR-VL et retourne le texte brut transcrit.'''
    model, processor = load_paddleocr_vl(device=device)

    pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    messages = [{
        "role": "user",
        "content": [{"type": "image", "image": pil_image}, {"type": "text", "text": prompt}],
    }]
    chat_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[chat_input], images=[pil_image], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=False)

    generated_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
    output_text = processor.batch_decode(
        generated_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0]
    return output_text.strip()


def ocr_document_with_paddleocr_vl(pages: List[np.ndarray]) -> str:
    '''OCRise toutes les pages (recto/verso) d'un document et concatène le texte.'''
    texts = []
    for i, page in enumerate(pages):
        try:
            texts.append(ocr_page_with_paddleocr_vl(page))
        except Exception as e:
            logger.error(f"Échec OCR PaddleOCR-VL page {i}: {e}")
            texts.append("")
    return "\n--- PAGE BREAK ---\n".join(texts)


## 9. Détection du pays d'émission

Utilisée pour décider si l'on applique un **schéma dédié Algérie** ou une **extraction générique**.
Priorité : code pays MRZ (`DZA`) > mots-clés dans le texte OCR (français/arabe) > défaut = pays inconnu.

In [ ]:
ALGERIA_KEYWORDS_FR = [
    "REPUBLIQUE ALGERIENNE", "REPUBLIQUE ALGERIENNE DEMOCRATIQUE",
    "WILAYA", "COMMUNE DE", "DAIRA", "ALGERIE",
]
# quelques mots-clés arabes fréquents sur les documents algériens
ALGERIA_KEYWORDS_AR = [
    "الجمهورية الجزائرية", "وزارة الداخلية", "بطاقة التعريف", "رخصة السياقة", "الجزائر",
]


def detect_country(mrz_pays: Optional[str], ocr_text: Optional[str]) -> str:
    '''Retourne 'DZ' si le document est identifié comme algérien, sinon 'AUTRE' (ou 'INCONNU').'''
    if mrz_pays and mrz_pays.upper() in ("DZA",):
        return "DZ"
    if mrz_pays and mrz_pays.upper() not in ("DZA", ""):
        return "AUTRE"

    if ocr_text:
        text_norm = unidecode(ocr_text).upper()
        for kw in ALGERIA_KEYWORDS_FR:
            if unidecode(kw).upper() in text_norm:
                return "DZ"
        for kw in ALGERIA_KEYWORDS_AR:
            if kw in ocr_text:
                return "DZ"
        return "AUTRE"

    return "INCONNU"


## 10. Schémas d'extraction dédiés — documents algériens (FR/AR)

Pour les documents identifiés comme **algériens**, on applique un schéma d'extraction **par type de document**
plutôt qu'un parsing générique : chaque schéma définit une liste de motifs `regex` (français **et** arabe) à
tester dans le texte OCR brut (issu soit de PaddleOCR-VL, soit — en complément — d'un OCR classique sur les
zones hors-MRZ). Le premier motif qui matche pour un champ donné est retenu.

> Ces regex sont un point de départ raisonnable ; à affiner avec de vrais échantillons (mise en page recto/verso
> CIN, permis, factures Sonelgaz / Algérie Télécom / attestation de résidence communale, etc.).

In [ ]:
DATE_PATTERN = r"(\d{1,2})[\/\.\-](\d{1,2})[\/\.\-](\d{2,4})"


def _std_date(day, month, year) -> Optional[str]:
    try:
        year = int(year)
        if year < 100:
            year += 2000 if year < 50 else 1900
        return datetime(year, int(month), int(day)).strftime("%Y-%m-%d")
    except (ValueError, TypeError):
        return None


ALGERIA_SCHEMAS = {
    "CIN": {
        "nom": [
            r"(?:Nom|NOM)\s*[:\-]?\s*([A-ZÀ-Ÿ' \-]+)",
            r"الاسم\s*[:\-]?\s*([\u0600-\u06FF ]+)",
        ],
        "prenom": [
            r"(?:Pr[ée]nom\(?s?\)?|PR[EÉ]NOM)\s*[:\-]?\s*([A-ZÀ-Ÿ' \-]+)",
            r"اللقب\s*[:\-]?\s*([\u0600-\u06FF ]+)",
        ],
        "date_naissance": [
            rf"(?:N[ée]\(?e?\)?\s+le|Date de naissance)\s*[:\-]?\s*{DATE_PATTERN}",
            rf"تاريخ الميلاد\s*[:\-]?\s*{DATE_PATTERN}",
        ],
        "date_expiration": [
            rf"(?:Valable jusqu.?au|Date d.?expiration|Expire le)\s*[:\-]?\s*{DATE_PATTERN}",
            rf"صالحة إلى غاية\s*[:\-]?\s*{DATE_PATTERN}",
        ],
        "numero_document": [
            r"(?:N[°ºo]\s*(?:de\s*)?(?:CIN|carte)?)\s*[:\-]?\s*([0-9]{6,})",
        ],
    },
    "PERMIS": {
        "nom": [
            r"(?:Nom|NOM)\s*[:\-]?\s*([A-ZÀ-Ÿ' \-]+)",
            r"اللقب\s*[:\-]?\s*([\u0600-\u06FF ]+)",
        ],
        "prenom": [
            r"(?:Pr[ée]nom\(?s?\)?)\s*[:\-]?\s*([A-ZÀ-Ÿ' \-]+)",
            r"الاسم\s*[:\-]?\s*([\u0600-\u06FF ]+)",
        ],
        "date_naissance": [
            rf"(?:N[ée]\(?e?\)?\s+le|Date de naissance)\s*[:\-]?\s*{DATE_PATTERN}",
            rf"تاريخ الميلاد\s*[:\-]?\s*{DATE_PATTERN}",
        ],
        "date_expiration": [
            rf"(?:Valable jusqu.?au|Date d.?expiration)\s*[:\-]?\s*{DATE_PATTERN}",
            rf"صالحة إلى غاية\s*[:\-]?\s*{DATE_PATTERN}",
        ],
        "numero_document": [
            r"(?:N[°ºo]\s*(?:de\s*)?permis)\s*[:\-]?\s*([0-9A-Z]{6,})",
        ],
    },
    "JUSTIFICATIF_DOMICILE": {
        "nom": [
            r"(?:M\.|Mme|Mr|Monsieur|Madame)\s+([A-ZÀ-Ÿ' \-]+)",
        ],
        "adresse": [
            r"(?:Adresse|Domicili[ée] à)\s*[:\-]?\s*(.+)",
            r"العنوان\s*[:\-]?\s*([\u0600-\u06FF0-9 ,]+)",
        ],
        "date_document": [
            rf"(?:Fait le|Date)\s*[:\-]?\s*{DATE_PATTERN}",
        ],
        "commune": [
            r"(?:Commune de|Daira de|Wilaya de)\s*[:\-]?\s*([A-ZÀ-Ÿ' \-]+)",
        ],
    },
}


def extract_with_schema(ocr_text: str, doc_type: str) -> Dict[str, Optional[str]]:
    '''Applique le schéma FR/AR correspondant au type de document algérien et retourne
    les champs trouvés (None si aucun motif ne matche).'''
    schema = ALGERIA_SCHEMAS.get(doc_type, {})
    result: Dict[str, Optional[str]] = {field: None for field in schema}

    for field_name, patterns in schema.items():
        for pattern in patterns:
            m = re.search(pattern, ocr_text, flags=re.IGNORECASE | re.MULTILINE)
            if m:
                if field_name in ("date_naissance", "date_expiration", "date_document") and len(m.groups()) == 3:
                    result[field_name] = _std_date(*m.groups())
                else:
                    result[field_name] = m.group(1).strip()
                break
    return result


def guess_algeria_doc_type(ocr_text: str, is_identite: bool) -> str:
    '''Détermine quel schéma algérien appliquer.'''
    if not is_identite:
        return "JUSTIFICATIF_DOMICILE"
    text_up = unidecode(ocr_text).upper()
    if "PERMIS" in text_up or "CONDUIRE" in text_up:
        return "PERMIS"
    return "CIN"


## 11. Extraction générique (documents non algériens, sans schéma)

Pour les documents étrangers, aucun schéma n'est disponible : on applique une extraction **best-effort**,
basée sur des regex de dates génériques et des heuristiques de proximité de mots-clés multilingues courants
(*name/nom/surname*, *date of birth/né le*, *expiry/expiration*, etc.).

In [ ]:
GENERIC_KEYWORDS = {
    "nom": [r"(?:surname|last\s*name|nom)\s*[:\-]?\s*([A-Za-zÀ-ÿ' \-]+)"],
    "prenom": [r"(?:given\s*name|first\s*name|pr[ée]nom)s?\s*[:\-]?\s*([A-Za-zÀ-ÿ' \-]+)"],
    "date_naissance": [rf"(?:date\s*of\s*birth|birth\s*date|n[ée]\(?e?\)?\s+le)\s*[:\-]?\s*{DATE_PATTERN}"],
    "date_expiration": [rf"(?:date\s*of\s*expiry|expiry\s*date|valid\s*until|expire[s]?)\s*[:\-]?\s*{DATE_PATTERN}"],
}


def extract_generic(ocr_text: str) -> Dict[str, Optional[str]]:
    result: Dict[str, Optional[str]] = {field: None for field in GENERIC_KEYWORDS}
    for field_name, patterns in GENERIC_KEYWORDS.items():
        for pattern in patterns:
            m = re.search(pattern, ocr_text, flags=re.IGNORECASE)
            if m:
                if field_name in ("date_naissance", "date_expiration") and len(m.groups()) == 3:
                    result[field_name] = _std_date(*m.groups())
                else:
                    result[field_name] = m.group(1).strip()
                break
    return result


## 12. Pipeline principal : traitement d'un document (identité ou domicile) puis d'un client complet

In [ ]:
@dataclass
class DocExtractionResult:
    client_id: str
    doc_role: str                       # "identite" ou "domicile"
    doc_type_detecte: Optional[str] = None   # CIN / PERMIS / PASSEPORT / JUSTIFICATIF_DOMICILE / GENERIQUE
    pays: Optional[str] = None               # DZ / AUTRE / INCONNU
    a_mrz: bool = False
    source_extraction: Optional[str] = None  # "MRZ" / "PADDLEOCR_VL"
    nom: Optional[str] = None
    prenom: Optional[str] = None
    date_naissance: Optional[str] = None
    date_expiration: Optional[str] = None
    champs_supplementaires: Dict = field(default_factory=dict)
    erreur: Optional[str] = None


def process_single_document(client_id: str, pdf_path: Optional[Path], doc_role: str) -> DocExtractionResult:
    result = DocExtractionResult(client_id=client_id, doc_role=doc_role)

    if pdf_path is None:
        result.erreur = "Fichier PDF introuvable dans le dossier client."
        return result

    try:
        pages = load_and_preprocess_pdf(pdf_path)
    except Exception as e:
        result.erreur = f"Échec conversion/prétraitement PDF : {e}"
        return result

    if not pages:
        result.erreur = "PDF vide (0 page)."
        return result

    # --- 1) Uniquement pour les documents d'identité : tenter la MRZ en premier -------------
    if doc_role == "identite":
        has_mrz, mrz_result = has_machine_readable_zone(pages, debug_name=f"{client_id}_{doc_role}")
        if has_mrz:
            parsed = parse_mrz(mrz_result)
            result.a_mrz = True
            result.source_extraction = "MRZ"
            result.nom = parsed.get("nom")
            result.prenom = parsed.get("prenom")
            result.date_naissance = parsed.get("date_naissance")
            result.date_expiration = parsed.get("date_expiration")
            result.doc_type_detecte = (
                "PASSEPORT" if parsed.get("type_document_probable") == "PASSEPORT" else "CIN_OU_PERMIS"
            )
            result.pays = "DZ" if parsed.get("pays_mrz") == "DZA" else (parsed.get("pays_mrz") or "INCONNU")
            result.champs_supplementaires["mrz_lines"] = parsed.get("mrz_lines_brutes")
            return result

    # --- 2) Pas de MRZ (ou justificatif de domicile) -> OCR via PaddleOCR-VL ------------------
    try:
        ocr_text = ocr_document_with_paddleocr_vl(pages)
    except Exception as e:
        result.erreur = f"Échec OCR PaddleOCR-VL : {e}"
        return result

    result.source_extraction = "PADDLEOCR_VL"
    result.champs_supplementaires["ocr_text_brut"] = ocr_text

    pays = detect_country(mrz_pays=None, ocr_text=ocr_text)
    result.pays = pays

    if pays == "DZ":
        doc_type = guess_algeria_doc_type(ocr_text, is_identite=(doc_role == "identite"))
        result.doc_type_detecte = doc_type
        extracted = extract_with_schema(ocr_text, doc_type)
    else:
        result.doc_type_detecte = "GENERIQUE"
        extracted = extract_generic(ocr_text)

    result.nom = extracted.get("nom")
    result.prenom = extracted.get("prenom")
    result.date_naissance = extracted.get("date_naissance")
    result.date_expiration = extracted.get("date_expiration") or extracted.get("date_document")
    for k, v in extracted.items():
        if k not in ("nom", "prenom", "date_naissance", "date_expiration", "date_document"):
            result.champs_supplementaires[k] = v

    return result


def process_client_folder(client_folder: Path) -> Dict[str, DocExtractionResult]:
    client_id = client_folder.name
    pdfs = find_target_pdfs(client_folder)

    identite_result = process_single_document(client_id, pdfs["identite"], "identite")
    domicile_result = process_single_document(client_id, pdfs["domicile"], "domicile")

    return {"identite": identite_result, "domicile": domicile_result}


## 13. Exécution du pipeline sur l'ensemble des dossiers clients

In [ ]:
def run_batch(extract_dir: Path) -> pd.DataFrame:
    client_folders = list_client_folders(extract_dir)
    rows = []

    for client_folder in tqdm(client_folders, desc="Traitement des dossiers clients"):
        try:
            docs = process_client_folder(client_folder)
        except Exception as e:
            logger.error(f"[{client_folder.name}] Échec complet du traitement : {e}")
            continue

        for role, res in docs.items():
            row = asdict(res)
            # on garde le texte OCR brut à part pour ne pas alourdir le tableau principal
            row["ocr_text_brut"] = row["champs_supplementaires"].pop("ocr_text_brut", None) \
                if row.get("champs_supplementaires") else None
            row["champs_supplementaires"] = json.dumps(row.get("champs_supplementaires") or {}, ensure_ascii=False)
            rows.append(row)

    df = pd.DataFrame(rows)
    return df


# Exécution réelle (décommenter) :
# extract_zip(ZIP_PATH, EXTRACT_DIR)
# df_extraction = run_batch(EXTRACT_DIR)
# df_extraction.to_csv(OUTPUT_DIR / "extraction_brute.csv", index=False)
# df_extraction.head(20)


## 14. Chargement de `tiers.csv` et consolidation par client

`tiers.csv` contient (au minimum) : `Date de naissance`, `id`, `tiers`, `nom abrege`, `date expiration du document`,
avec des dates déjà au format `YYYY-MM-DD`. On consolide d'abord `df_extraction` (une ligne par document,
donc 2 lignes par client) en **une ligne par client** (`identite` + `domicile` mis à plat), avant de joindre.

In [ ]:
def pivot_extraction_per_client(df_extraction: pd.DataFrame) -> pd.DataFrame:
    '''Passe de 'une ligne par document' à 'une ligne par client', avec les infos d'identité
    en colonnes principales (utilisées pour le matching) et celles du domicile préfixées.'''
    id_docs = df_extraction[df_extraction["doc_role"] == "identite"].copy()
    dom_docs = df_extraction[df_extraction["doc_role"] == "domicile"].copy()

    id_docs = id_docs.rename(columns={
        c: c for c in id_docs.columns
    }).add_suffix("_identite")
    id_docs = id_docs.rename(columns={"client_id_identite": "id"})

    dom_docs = dom_docs.add_suffix("_domicile")
    dom_docs = dom_docs.rename(columns={"client_id_domicile": "id"})

    merged = pd.merge(id_docs, dom_docs, on="id", how="outer")
    return merged


def load_tiers(tiers_csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(tiers_csv_path, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    expected_cols = ["Date de naissance", "id", "tiers", "nom abrege", "date expiration du document"]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        logger.warning(f"Colonnes attendues absentes de tiers.csv : {missing}")

    # Les dates sont déjà en YYYY-MM-DD selon l'énoncé -> on valide/normalise quand même
    for col in ["Date de naissance", "date expiration du document"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: _safe_parse_iso_date(x))

    df["id"] = df["id"].astype(str).str.strip()
    return df


def _safe_parse_iso_date(value) -> Optional[str]:
    if pd.isna(value) or not str(value).strip():
        return None
    value = str(value).strip()
    try:
        return dateparser.parse(value, yearfirst=True, dayfirst=False).strftime("%Y-%m-%d")
    except (ValueError, TypeError):
        return None


## 15. Matching nom/prénom robuste

`nom abrege` peut être `NOM PRENOM` **ou** `PRENOM NOM` selon les tiers. On compare donc les **jeux de mots**
(insensibles à l'ordre, à la casse et aux accents) entre `nom abrege` (tiers.csv) et `nom` + `prenom` extraits,
via un score de similarité (`rapidfuzz.token_sort_ratio`, qui est justement conçu pour ignorer l'ordre des mots).

In [ ]:
def normalize_name_tokens(text: Optional[str]) -> str:
    '''Normalise une chaîne nom/prénom : suppression accents, majuscules, espaces multiples,
    tri des mots par ordre alphabétique (pour neutraliser l'ordre nom/prénom).'''
    if not text or pd.isna(text):
        return ""
    text = unidecode(str(text)).upper()
    text = re.sub(r"[^A-Z ]", " ", text)
    tokens = sorted(t for t in text.split() if t)
    return " ".join(tokens)


def name_match_score(nom_abrege: Optional[str], nom_extrait: Optional[str], prenom_extrait: Optional[str]) -> float:
    '''Score de similarité (0-100) entre 'nom abrege' (tiers.csv) et nom+prénom extraits du document,
    indépendamment de l'ordre nom/prénom.'''
    tiers_norm = normalize_name_tokens(nom_abrege)
    extrait_norm = normalize_name_tokens(f"{nom_extrait or ''} {prenom_extrait or ''}")
    if not tiers_norm or not extrait_norm:
        return 0.0
    return fuzz.token_sort_ratio(tiers_norm, extrait_norm)


## 16. Jointure `extraction` ⨝ `tiers.csv` et détection des incohérences

Seuils par défaut (ajustables) :
- `NAME_MATCH_THRESHOLD` : score de similarité nom/prénom en-dessous duquel on signale une incohérence.
- Les dates sont comparées en **égalité stricte** après normalisation `YYYY-MM-DD` (un enregistrement `None`
  d'un côté est traité comme "non vérifiable", pas comme une incohérence).

In [ ]:
NAME_MATCH_THRESHOLD = 85.0


def build_incoherence_report(df_extraction: pd.DataFrame, df_tiers: pd.DataFrame) -> pd.DataFrame:
    df_client = pivot_extraction_per_client(df_extraction)
    merged = pd.merge(df_client, df_tiers, on="id", how="inner", suffixes=("", "_tiers"))

    rows = []
    for _, r in merged.iterrows():
        nom_abrege = r.get("nom abrege")
        nom_extrait = r.get("nom_identite")
        prenom_extrait = r.get("prenom_identite")
        score_nom = name_match_score(nom_abrege, nom_extrait, prenom_extrait)

        dob_tiers = r.get("Date de naissance")
        dob_extrait = r.get("date_naissance_identite")
        dob_match = (dob_tiers == dob_extrait) if (dob_tiers and dob_extrait) else None

        exp_tiers = r.get("date expiration du document")
        exp_extrait = r.get("date_expiration_identite")
        exp_match = (exp_tiers == exp_extrait) if (exp_tiers and exp_extrait) else None

        incoherences = []
        if score_nom < NAME_MATCH_THRESHOLD:
            incoherences.append(f"NOM/PRENOM (score={score_nom:.0f})")
        if dob_match is False:
            incoherences.append(f"DATE_NAISSANCE (tiers={dob_tiers} / extrait={dob_extrait})")
        if exp_match is False:
            incoherences.append(f"DATE_EXPIRATION (tiers={exp_tiers} / extrait={exp_extrait})")
        if r.get("erreur_identite"):
            incoherences.append(f"ERREUR_EXTRACTION_IDENTITE: {r.get('erreur_identite')}")
        if r.get("erreur_domicile"):
            incoherences.append(f"ERREUR_EXTRACTION_DOMICILE: {r.get('erreur_domicile')}")

        rows.append({
            "id": r.get("id"),
            "tiers": r.get("tiers"),
            "nom_abrege_tiers": nom_abrege,
            "nom_extrait": nom_extrait,
            "prenom_extrait": prenom_extrait,
            "score_matching_nom": round(score_nom, 1),
            "date_naissance_tiers": dob_tiers,
            "date_naissance_extraite": dob_extrait,
            "date_expiration_tiers": exp_tiers,
            "date_expiration_extraite": exp_extrait,
            "pays_document": r.get("pays_identite"),
            "a_mrz": r.get("a_mrz_identite"),
            "source_extraction": r.get("source_extraction_identite"),
            "nb_incoherences": len(incoherences),
            "detail_incoherences": " | ".join(incoherences) if incoherences else "RAS",
            "statut": "A_VERIFIER" if incoherences else "OK",
        })

    report_df = pd.DataFrame(rows)

    # clients présents dans l'extraction mais absents de tiers.csv (ou inversement) -> à signaler aussi
    ids_extraction = set(df_client["id"])
    ids_tiers = set(df_tiers["id"])
    only_in_extraction = ids_extraction - ids_tiers
    only_in_tiers = ids_tiers - ids_extraction
    if only_in_extraction:
        logger.warning(f"{len(only_in_extraction)} id(s) présents dans l'extraction mais absents de tiers.csv.")
    if only_in_tiers:
        logger.warning(f"{len(only_in_tiers)} id(s) présents dans tiers.csv mais absents de l'extraction.")

    return report_df.sort_values("nb_incoherences", ascending=False)


## 17. Export du rapport final

In [ ]:
# Exécution réelle de bout en bout (décommenter) :
#
# extract_zip(ZIP_PATH, EXTRACT_DIR)
# df_extraction = run_batch(EXTRACT_DIR)
# df_extraction.to_csv(OUTPUT_DIR / "extraction_brute.csv", index=False)
#
# df_tiers = load_tiers(TIERS_CSV_PATH)
#
# report_df = build_incoherence_report(df_extraction, df_tiers)
# report_df.to_csv(OUTPUT_DIR / "rapport_incoherences.csv", index=False)
# report_df.to_excel(OUTPUT_DIR / "rapport_incoherences.xlsx", index=False)
#
# n_total = len(report_df)
# n_ok = (report_df["statut"] == "OK").sum()
# print(f"{n_ok}/{n_total} clients sans incohérence détectée.")
# report_df.head(30)

print("Pipeline prêt. Décommenter les cellules d'exécution une fois ZIP_PATH / TIERS_CSV_PATH renseignés.")


## 18. Test unitaire de validation — exemple réel (CIN biométrique algérienne)

Vérification du parsing MRZ TD1 (`parse_mrz_td1`) sur l'exemple concret fourni :

- Nom / Prénom affichés sur la carte : **El Hachimi / Said**
- Ligne 1 MRZ : `IDDZA1512345670<<<<<<<<<<<<<<<`
- Ligne 2 MRZ : `8406226M2509083DZA<<<<<<<<<<<<<0`
- Ligne 3 MRZ : `EL<HACHIMI<<SAID<<<<<<<<<<<<<<<<`

Attendu : date de naissance `1984-06-22` (zone encadrée en rouge sur l'exemple = `840622`), date d'expiration
`2025-09-08` (cohérente avec le "2025" affiché visuellement sur la carte), nom `EL HACHIMI`, prénom `SAID`.

In [ ]:
# Cellule d'auto-test : à exécuter telle quelle (ne nécessite ni zip ni modèle) pour valider
# que le parsing MRZ produit bien les résultats attendus sur un exemple réel connu.

_test_lines_cin = [
    "IDDZA1512345670<<<<<<<<<<<<<<<",
    "8406226M2509083DZA<<<<<<<<<<<<<0",
    "EL<HACHIMI<<SAID<<<<<<<<<<<<<<<<",
]

_parsed_test = parse_mrz({"lines": _test_lines_cin})

print("Résultat du parsing :")
for k, v in _parsed_test.items():
    if k != "mrz_lines_brutes":
        print(f"  {k}: {v}")

assert _parsed_test["nom"] == "EL HACHIMI", f"Nom inattendu : {_parsed_test['nom']}"
assert _parsed_test["prenom"] == "SAID", f"Prénom inattendu : {_parsed_test['prenom']}"
assert _parsed_test["date_naissance"] == "1984-06-22", f"Date naissance inattendue : {_parsed_test['date_naissance']}"
assert _parsed_test["date_expiration"] == "2025-09-08", f"Date expiration inattendue : {_parsed_test['date_expiration']}"
assert _parsed_test["pays_mrz"] == "DZA", f"Pays inattendu : {_parsed_test['pays_mrz']}"
assert _parsed_test["type_document_probable"] == "CIN_ou_PERMIS"

print("\\n✅ Test réussi : le parsing MRZ TD1 reproduit exactement les valeurs attendues sur l'exemple réel.")


## 19. Limites connues & pistes d'amélioration

- **Qualité des regex des schémas algériens** : à valider/enrichir sur un échantillon réel (CIN recto/verso,
  permis, factures/attestations de domicile) — les libellés exacts (accents, majuscules, sauts de ligne OCR)
  varient beaucoup.
- **Prompt PaddleOCR-VL** : peut être spécialisé par type de document (ex. demander explicitement une sortie
  structurée JSON) pour réduire le post-traitement regex, si le modèle le supporte bien en pratique.
- **Robustesse du split de siècle des dates MRZ** (`YY` → 19xx/20xx) : la règle actuelle est heuristique ; si les
  dossiers contiennent des personnes très âgées ou des documents très anciens, affiner le seuil.
- **`PassportEye` sur photocopies très dégradées** : peut manquer la zone MRZ malgré le prétraitement — dans ce
  cas le document bascule automatiquement sur PaddleOCR-VL, ce qui est le comportement voulu, mais il peut être
  utile de logguer spécifiquement ces cas pour un contrôle qualité manuel.
- **GPU** : PaddleOCR-VL est un modèle vision-langage ; sur gros volumes, exécuter sur GPU (`USE_GPU=1`) et
  envisager un traitement par batch pour accélérer l'inférence.
- **Contrôle qualité humain** : toute ligne `A_VERIFIER` (ou tout document en erreur d'extraction) devrait être
  routée vers une revue manuelle avant recertification finale — ce notebook est une aide à la décision, pas un
  automate de décision KYC autonome.
